# Criticism Detection and Analysis

This notebook demonstrates how to detect and analyze criticism in customer support text data. You will:
- Download a real-world dataset
- Explore and visualize criticism patterns
- Build and evaluate a simple machine learning model for criticism detection

**Instructions:**
- Run the first cell to download the dataset.
- Make sure you have the required libraries installed (see below).
- Follow each section to complete the analysis.

In [12]:
# from datasets import load_dataset
ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")

In [13]:
# Print the first 10 dataset instances to inspect structure
for i in range(10):
    print(ds[i])


{'flags': 'B', 'instruction': 'question about cancelling order {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}
{'flags': 'BQZ', 'instruction': 'i have a question about cancelling oorder {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've been informed that you have a question about canceling order {{Order Number}}. I'm here to assist you! Please go ahead and let me know what specific question you have, and I'll provide you with all the information and guidance you need. Your satisfaction is my top priority."}
{'flags': 'BLQZ', 'instruction': 'i need help cancelling puchase {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I can sense that you're seeking assistance with canceling your 

In [4]:
import sys

# This checks if they are installed and prints their version
try:
    import dotenv
    import openai
    print("python-dotenv is installed")
    print("openai is installed")
except ImportError as e:
    print(f"Missing package: {e}")

python-dotenv is installed
openai is installed


In [6]:
# Install required packages if not already present
try:
    import jinja2, datasets
    print("jinja2 and datasets are ready")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "jinja2", "datasets", "-q"])
    print("Packages installed — please restart the kernel and re-run")


jinja2 and datasets are ready


In [8]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Add it to .env")

client = OpenAI()
MODEL = "gpt-4.1-mini"
print("Client ready")


Client ready


In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score

## Call 1: Extract — `promptv1.j2`

**Goal:** Given a raw customer support request, extract the intent (short verb phrase) and a list of observable symptoms.

**Input:** `request`  
**Output:** `{ "intent": "...", "symptoms": [...] }`


In [14]:
import json
from jinja2 import Template

# Load all 4 prompt templates
templates = {}
for version in ["promptv1", "promptv2", "promptv3", "promptv4"]:
    with open(f"{version}.j2", "r") as f:
        templates[version] = Template(f.read())

# Prepare the first 10 requests from the dataset
N = 10
# Use the correct field for customer request (update after inspecting ds[i])
requests = [ds[i].get("response", "") for i in range(N)]

# Render prompts for Call 1 (extraction)
prompts_v1 = [templates["promptv1"].render(request=req) for req in requests]

print(f"Loaded {len(templates)} templates: {list(templates.keys())}")
print(f"Prepared {len(requests)} requests")
print(f"\nExample prompt (first item):\n{prompts_v1[0]}")


Loaded 4 templates: ['promptv1', 'promptv2', 'promptv3', 'promptv4']
Prepared 10 requests

Example prompt (first item):
You are a support-ticket triage assistant.

Given the customer request below, extract:
- intent: the customer's primary goal (short verb phrase)
- symptoms: list of concrete, observable problems

Respond in JSON with keys "intent" and "symptoms".

Customer request:
I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.


In [ ]:
def safe_parse_json(text):
    """Extract and parse the first JSON object found in the LLM response."""
    try:
        start = text.find('{')
        end = text.rfind('}') + 1
        if start != -1 and end > start:
            return json.loads(text[start:end])
    except Exception:
        pass
    return None

# Call 1: Extract with promptv1
responses_v1_raw = []
for prompt in prompts_v1:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    responses_v1_raw.append(response.choices[0].message.content)

extractions_v1 = [safe_parse_json(r) for r in responses_v1_raw]

print("--- Call 1: Extractions (promptv1) ---")
for i, (req, ext) in enumerate(zip(requests, extractions_v1)):
    print(f"[{i}] {req}")
    print(f"  → {ext}")
    print()


--- Call 1: Extractions (promptv1) ---
[0] I've understood you have a question regarding canceling orde
  → {'intent': 'ask about canceling order', 'symptoms': []}

[1] I've been informed that you have a question about canceling 
  → {'intent': 'ask about order cancellation', 'symptoms': []}

[2] I can sense that you're seeking assistance with canceling yo
  → {'intent': 'cancel purchase', 'symptoms': ['needs guidance on how to cancel a purchase', 'uncertainty or difficulty with the cancellation process']}

[3] I understood that you need assistance with canceling your pu
  → {'intent': 'cancel purchase', 'symptoms': []}

[4] I'm sensitive to the fact that you're facing financial diffi
  → {'intent': 'cancel purchase', 'symptoms': ['customer wants to cancel their order', 'customer is facing financial difficulties', 'needs guidance on how to cancel the order online']}

[5] Of course, I'm here to assist you in canceling your order wi
  → {'intent': 'cancel order', 'symptoms': []}

[6] I p

## Call 2: Judge — `promptv2.j2`

**Goal:** A strict quality reviewer checks each v1 extraction against the original request — looking for inaccurate intents, missing or hallucinated symptoms — and produces a corrected version.

**Input:** `request` + `extraction` (output of Call 1)  
**Output:** `{ "review": "...", "corrected_intent": "...", "corrected_symptoms": [...] }`


In [ ]:
# Call 2: Judge — reviews v1 extractions and produces corrections
responses_v2_raw = []
for req, ext in zip(requests, extractions_v1):
    ext_str = json.dumps(ext) if ext else "{}"
    prompt = templates["promptv2"].render(request=req, extraction=ext_str)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    responses_v2_raw.append(response.choices[0].message.content)

reviews_v2 = [safe_parse_json(r) for r in responses_v2_raw]

print("--- Call 2: Reviews (promptv2) ---")
for i, review in enumerate(reviews_v2):
    if review:
        print(f"[{i}] Review: {review.get('review', '')}")
        print(f"  corrected_intent:   {review.get('corrected_intent', '—')}")
        print(f"  corrected_symptoms: {review.get('corrected_symptoms', [])}")
        print()


--- Call 2: Reviews (promptv2) ---
[0] Review: The intent is broadly accurate as the user mentions a question regarding canceli
  corrected_intent:   offer assistance regarding order cancellation
  corrected_symptoms: []

[1] Review: The intent 'ask about order cancellation' is generally accurate but somewhat pre
  corrected_intent:   inquiry about potential order cancellation question
  corrected_symptoms: []

[2] Review: The intent 'cancel purchase' is accurate and appropriately specific. The symptom
  corrected_intent:   cancel purchase
  corrected_symptoms: ['seeking help to cancel a purchase', 'requesting step-by-step guidance for cancellation', 'expressing inconvenience caused by the cancellation process']

[3] Review: The intent 'cancel purchase' is accurate but could be more specific by including
  corrected_intent:   cancel purchase order
  corrected_symptoms: ['need assistance with cancellation', 'order number provided', 'inconvenience caused']

[4] Review: The intent 'cancel

## Call 3: Improve — `promptv3.j2`

**Goal:** A feedback-loop assistant critiques the corrected v2 extraction and produces an even better version, pushing for specificity and completeness.

**Input:** `request` + `extraction` (corrected output from Call 2)  
**Output:** `{ "critique": "...", "better_extraction": { "intent": "...", "symptoms": [...] } }`


In [17]:
def corrected_from_v2(review):
    """Reconstruct a flat extraction dict from the v2 judge output."""
    if review:
        return {
            "intent": review.get("corrected_intent", ""),
            "symptoms": review.get("corrected_symptoms", [])
        }
    return {}

# Call 3: Improve — takes v2 corrected extraction and produces a better one
responses_v3_raw = []
for req, review in zip(requests, reviews_v2):
    corrected = corrected_from_v2(review)
    ext_str = json.dumps(corrected)
    prompt = templates["promptv3"].render(request=req, extraction=ext_str)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    responses_v3_raw.append(response.choices[0].message.content)

improvements_v3 = [safe_parse_json(r) for r in responses_v3_raw]

print("--- Call 3: Improvements (promptv3) ---")
for i, imp in enumerate(improvements_v3):
    if imp:
        be = imp.get("better_extraction", {})
        print(f"[{i}] Critique: {str(imp.get('critique', ''))[:80]}")
        print(f"  better_intent:   {be.get('intent', '—')}")
        print(f"  better_symptoms: {be.get('symptoms', [])}")
        print()


--- Call 3: Improvements (promptv3) ---
[0] Critique: The extraction correctly identifies the general intent as offering assistance re
  better_intent:   invite customer to ask question about order cancellation
  better_symptoms: ['no specific question asked yet', 'mention of order number placeholder']

[1] Critique: The extraction inaccurately interprets the intent. The original message is not a
  better_intent:   offer assistance with order cancellation query
  better_symptoms: []

[2] Critique: The extraction accurately identifies the intent as 'cancel purchase' and recogni
  better_intent:   cancel purchase
  better_symptoms: ['requests to cancel a purchase using order number', 'seeks step-by-step instructions for cancellation', 'requires information about account login and order location', 'needs contact details for additional support if difficulties arise']

[3] Critique: The extraction identifies the primary intent of canceling a purchase order corre
  better_intent:   cancel pu

## Call 4: Final Judge — `promptv4.j2`

**Goal:** An independent final judge evaluates the fully-improved extraction. It does **not** correct anything — it only scores (0–10) and suggests remaining improvements.

**Input:** `request` + `extraction` (better version from Call 3)  
**Output:** `{ "judgement": "...", "score": 0-10, "suggestions": [...] }`


In [18]:
def better_from_v3(imp):
    """Extract the better_extraction dict from the v3 improver output."""
    if imp:
        return imp.get("better_extraction", {})
    return {}

# Call 4: Final Judge — scores the fully-improved extraction
responses_v4_raw = []
for req, imp in zip(requests, improvements_v3):
    better = better_from_v3(imp)
    ext_str = json.dumps(better)
    prompt = templates["promptv4"].render(request=req, extraction=ext_str)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    responses_v4_raw.append(response.choices[0].message.content)

judgements_v4 = [safe_parse_json(r) for r in responses_v4_raw]

print("--- Call 4: Final Judgements (promptv4) ---")
for i, j in enumerate(judgements_v4):
    if j:
        score = j.get("score", "?")
        print(f"[{i}] Score: {score}/10")
        print(f"  Judgement:   {str(j.get('judgement', ''))[:80]}")
        print(f"  Suggestions: {j.get('suggestions', [])}")
        print()


--- Call 4: Final Judgements (promptv4) ---
[0] Score: 8/10
  Judgement:   The extraction accurately captures the intent as an invitation rather than a dir
  Suggestions: ["Refine the intent to emphasize that this is a support agent's prompt rather than a customer request.", "Remove or clarify the 'mention of order number placeholder' as it is not a symptom of the issue but part of the template.", "Consider adding a symptom indicating 'customer has not yet asked a question' to clearly reflect the current session state."]

[1] Score: 8/10
  Judgement:   The proposed extraction correctly infers the intent is related to assisting with
  Suggestions: ["Refine intent to indicate this is an offer to assist with order cancellation questions rather than a query itself, e.g., 'offer assistance regarding order cancellation inquiry'.", 'Confirm no symptoms should be extracted as none are present in the original request.', 'Ensure that the intent captures the proactive support nature rather than a

## Results: Pipeline Comparison

The table below shows how the intent extraction evolved through all 4 calls for each of the 10 dataset items, and the final quality score assigned by the independent judge.


In [ ]:
# Build a side-by-side comparison table across all 4 calls
rows = []
for i in range(len(requests)):
    v1  = extractions_v1[i]  or {}
    v2  = reviews_v2[i]      or {}
    v3  = improvements_v3[i] or {}
    v4  = judgements_v4[i]   or {}
    better = v3.get("better_extraction", {})
    rows.append({
        "request":            requests[i][:45],
        "v1_intent":          v1.get("intent", "—"),
        "v2_corrected_intent": v2.get("corrected_intent", "—"),
        "v3_better_intent":   better.get("intent", "—"),
        "v4_score":           v4.get("score", "—"),
    })

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 60)
display(df)
